# GMA / FDIIR — Exploração do Espaço de Soluções

**Estudo Dirigido — Mestrado FDIIR**  

Execução dos quatro métodos de seleção morfológica sobre a CCM:  
Scoring linear ponderado · Ideal point · Pareto-based MA · HMMD  



## Setup

In [21]:
import numpy as np
import pandas as pd
import itertools
from scipy.stats import spearmanr

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

### Dados

As constantes do projeto.


#### Parâmetros morfológicos e design alternatives (DAs)

In [22]:
# Módulos e seus valores (IDs internos)
MODULES = ['D', 'I', 'ID', 'R']

MODULE_VALUES = {
    'D':  ['D1', 'D2', 'D3', 'D4'],
    'I':  ['I1', 'I2', 'I3', 'I4'],
    'ID': ['ID1', 'ID2', 'ID3'],
    'R':  ['R1', 'R2', 'R3']
}

# Nomes descritivos das DAs
DA_NAMES = {
    'D1':  'model-based',
    'D2':  'signal-based',
    'D3':  'data-driven',
    'D4':  'active diagnosis',
    'I1':  'output comparison',
    'I2':  'voting',
    'I3':  'dependency modeling',
    'I4':  'adaptive filters',
    'ID1': 'rule-based',
    'ID2': 'statistical',
    'ID3': 'ML identification',
    'R1':  'exclusion',
    'R2':  'weight reconfiguration',
    'R3':  'bypass'
}

# Critérios
CRITERIA = ['C1', 'C2', 'C3', 'C4', 'C5']
CRITERIA_NAMES = {
    'C1': 'Interpretabilidade',
    'C2': 'Custo computacional (inv.)',
    'C3': 'Latência (inv.)',
    'C4': 'Maturidade tecnológica',
    'C5': 'Cobertura de falhas'
}

print("Módulos:", MODULES)
print("Total DAs:", sum(len(v) for v in MODULE_VALUES.values()))
print("Espaço bruto:", np.prod([len(v) for v in MODULE_VALUES.values()]), "configurações")

Módulos: ['D', 'I', 'ID', 'R']
Total DAs: 14
Espaço bruto: 144 configurações


#### Vetor A Bruto

In [23]:
# Vetor A: scores C1–C5 por DA
# Ordem colunas: C1, C2, C3, C4, C5
VETOR_A_RAW = {
    # Módulo D
    'D1':  [5, 3, 4, 5, 4],
    'D2':  [4, 4, 5, 5, 3],
    'D3':  [1, 1, 2, 3, 4],
    'D4':  [4, 3, 2, 2, 3],
    # Módulo I
    'I1':  [5, 3, 4, 5, 4],
    'I2':  [5, 3, 4, 5, 3],
    'I3':  [4, 2, 2, 3, 4],
    'I4':  [3, 3, 4, 4, 4],
    # Módulo ID
    'ID1': [5, 5, 5, 5, 3],
    'ID2': [3, 3, 4, 5, 4],
    'ID3': [1, 2, 2, 3, 3],
    # Módulo R
    'R1':  [5, 5, 5, 5, 3],
    'R2':  [4, 3, 3, 4, 4],
    'R3':  [5, 4, 5, 4, 4]
}

# DataFrame do Vetor A
vetor_a = pd.DataFrame(VETOR_A_RAW, index=CRITERIA).T
vetor_a.index.name = 'DA'
vetor_a['Sigma'] = vetor_a.sum(axis=1)

print("Vetor A — Scores consolidados")
print(vetor_a.to_string())

Vetor A — Scores consolidados
     C1  C2  C3  C4  C5  Sigma
DA                            
D1    5   3   4   5   4     21
D2    4   4   5   5   3     21
D3    1   1   2   3   4     11
D4    4   3   2   2   3     14
I1    5   3   4   5   4     21
I2    5   3   4   5   3     20
I3    4   2   2   3   4     15
I4    3   3   4   4   4     18
ID1   5   5   5   5   3     23
ID2   3   3   4   5   4     19
ID3   1   2   2   3   3     11
R1    5   5   5   5   3     23
R2    4   3   3   4   4     18
R3    5   4   5   4   4     22


### CCM — Cross-Consistency Matrix graduada

Escala {0.0, 0.5, 1.0}.  
Resultado de 3 passadas multi-LLM + fontes primárias.

In [24]:
# CCM — 6 blocos de compatibilidade par-a-par
# Estrutura: CCM[(da1, da2)] = score {0.0, 0.5, 1.0}
# Simétrico: CCM[(a,b)] == CCM[(b,a)]

CCM_RAW = {
    # B1: D × R
    ('D1', 'R1'): 1.0, ('D1', 'R2'): 1.0, ('D1', 'R3'): 1.0,
    ('D2', 'R1'): 1.0, ('D2', 'R2'): 1.0, ('D2', 'R3'): 1.0,
    ('D3', 'R1'): 1.0, ('D3', 'R2'): 1.0, ('D3', 'R3'): 1.0,
    ('D4', 'R1'): 0.5, ('D4', 'R2'): 1.0, ('D4', 'R3'): 0.5,

    # B2: D × ID
    ('D1', 'ID1'): 1.0, ('D1', 'ID2'): 1.0, ('D1', 'ID3'): 1.0,
    ('D2', 'ID1'): 1.0, ('D2', 'ID2'): 1.0, ('D2', 'ID3'): 1.0,
    ('D3', 'ID1'): 0.5, ('D3', 'ID2'): 1.0, ('D3', 'ID3'): 1.0,
    ('D4', 'ID1'): 0.5, ('D4', 'ID2'): 0.5, ('D4', 'ID3'): 0.5,

    # B3: D × I
    ('D1', 'I1'): 1.0, ('D1', 'I2'): 1.0, ('D1', 'I3'): 1.0, ('D1', 'I4'): 1.0,
    ('D2', 'I1'): 1.0, ('D2', 'I2'): 1.0, ('D2', 'I3'): 1.0, ('D2', 'I4'): 1.0,
    ('D3', 'I1'): 0.5, ('D3', 'I2'): 1.0, ('D3', 'I3'): 1.0, ('D3', 'I4'): 1.0,
    ('D4', 'I1'): 0.5, ('D4', 'I2'): 0.5, ('D4', 'I3'): 1.0, ('D4', 'I4'): 0.5,

    # B4: I × R
    ('I1', 'R1'): 1.0, ('I1', 'R2'): 1.0, ('I1', 'R3'): 1.0,
    ('I2', 'R1'): 1.0, ('I2', 'R2'): 1.0, ('I2', 'R3'): 1.0,
    ('I3', 'R1'): 1.0, ('I3', 'R2'): 1.0, ('I3', 'R3'): 1.0,
    ('I4', 'R1'): 1.0, ('I4', 'R2'): 1.0, ('I4', 'R3'): 0.5,

    # B5: I × ID
    ('I1', 'ID1'): 1.0, ('I1', 'ID2'): 1.0, ('I1', 'ID3'): 1.0,
    ('I2', 'ID1'): 1.0, ('I2', 'ID2'): 1.0, ('I2', 'ID3'): 1.0,
    ('I3', 'ID1'): 1.0, ('I3', 'ID2'): 1.0, ('I3', 'ID3'): 1.0,
    ('I4', 'ID1'): 1.0, ('I4', 'ID2'): 1.0, ('I4', 'ID3'): 1.0,

    # B6: ID × R
    ('ID1', 'R1'): 1.0, ('ID1', 'R2'): 1.0, ('ID1', 'R3'): 1.0,
    ('ID2', 'R1'): 1.0, ('ID2', 'R2'): 1.0, ('ID2', 'R3'): 0.5,
    ('ID3', 'R1'): 1.0, ('ID3', 'R2'): 1.0, ('ID3', 'R3'): 1.0,
}

# Tornar simétrico
CCM = {}
for (a, b), v in CCM_RAW.items():
    CCM[(a, b)] = v
    CCM[(b, a)] = v

def get_ccm(da1, da2):
    #Retorna score CCM para um par de DAs. Retorna 1.0 se mesmo módulo (irrelevante).
    if da1 == da2:
        return 1.0
    return CCM.get((da1, da2), CCM.get((da2, da1), None))

# Verificação
n_cells = len(CCM_RAW)
n_05 = sum(1 for v in CCM_RAW.values() if v == 0.5)
n_10 = sum(1 for v in CCM_RAW.values() if v == 1.0)
n_00 = sum(1 for v in CCM_RAW.values() if v == 0.0)

print(f"CCM carregada: {n_cells} células")
print(f"  1.0 (compatível):     {n_10} ({100*n_10/n_cells:.1f}%)")
print(f"  0.5 (condicional):    {n_05} ({100*n_05/n_cells:.1f}%)")
print(f"  0.0 (incompatível):   {n_00} ({100*n_00/n_cells:.1f}%)")

CCM carregada: 73 células
  1.0 (compatível):     61 (83.6%)
  0.5 (condicional):    12 (16.4%)
  0.0 (incompatível):   0 (0.0%)


### 1.4 Cenários SC1–SC4
 
Pesos de módulo homogêneos em todos os cenários (w_D = w_I = w_ID = w_R = 0.25).  
Diferenciação exclusivamente pelos pesos dos critérios (w_C1–w_C5).

In [25]:
# Cenários: pesos dos módulos e dos critérios
SCENARIOS = {
    'SC1': {
        'name': 'Uniforme',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .20, 'C2': .20, 'C3': .20, 'C4': .20, 'C5': .20}
    },
    'SC2': {
        'name': 'Segurança / certificação',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .35, 'C2': .10, 'C3': .10, 'C4': .35, 'C5': .10}
    },
    'SC3': {
        'name': 'Cobertura / desempenho',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .10, 'C2': .20, 'C3': .20, 'C4': .10, 'C5': .40}
    },
    'SC4': {
        'name': 'Eficiência de recursos',
        'module_weights': {'D': .25, 'I': .25, 'ID': .25, 'R': .25},
        'criteria_weights': {'C1': .10, 'C2': .35, 'C3': .35, 'C4': .10, 'C5': .10}
    }
}

# Verificação: pesos dos critérios somam 1.0
for sc_id, sc in SCENARIOS.items():
    total = sum(sc['criteria_weights'].values())
    assert abs(total - 1.0) < 1e-9, f"{sc_id}: pesos dos critérios somam {total}"

# Exibição
rows = []
for sc_id, sc in SCENARIOS.items():
    row = {'Cenário': sc_id, 'Semântica': sc['name']}
    row.update({f'w_{k}': v for k, v in sc['module_weights'].items()})
    row.update({f'w_{k}': v for k, v in sc['criteria_weights'].items()})
    rows.append(row)

df_scenarios = pd.DataFrame(rows).set_index('Cenário')
print("Cenários SC1-SC4")
print(df_scenarios.to_string())

Cenários SC1-SC4
                        Semântica   w_D   w_I  w_ID   w_R  w_C1  w_C2  w_C3  w_C4  w_C5
Cenário                                                                                
SC1                      Uniforme 0.250 0.250 0.250 0.250 0.200 0.200 0.200 0.200 0.200
SC2      Segurança / certificação 0.250 0.250 0.250 0.250 0.350 0.100 0.100 0.350 0.100
SC3        Cobertura / desempenho 0.250 0.250 0.250 0.250 0.100 0.200 0.200 0.100 0.400
SC4        Eficiência de recursos 0.250 0.250 0.250 0.250 0.100 0.350 0.350 0.100 0.100


## Espaco de Solucoes


### Geracao do produto cartesiano D x I x ID x R


In [26]:
def generate_configurations():
    """
    Gera todas as 144 configuracoes do produto cartesiano D x I x ID x R.
    Retorna lista de dicts: {'D': da_d, 'I': da_i, 'ID': da_id, 'R': da_r}
    """
    configs = []
    for d, i, id_, r in itertools.product(
        MODULE_VALUES['D'],
        MODULE_VALUES['I'],
        MODULE_VALUES['ID'],
        MODULE_VALUES['R']
    ):
        configs.append({'D': d, 'I': i, 'ID': id_, 'R': r})
    return configs


def compute_w(config):
    """
    Calcula w(S) = minimo das compatibilidades par-a-par entre DAs
    de modulos diferentes na configuracao S.
    Fonte: Levin (2012), Secao 3.7 -- N(S) = (w(S); n(S))
    Os 6 pares correspondem aos blocos B1-B6 da CCM E5.
    """
    das = [config['D'], config['I'], config['ID'], config['R']]
    scores = []
    for da1, da2 in itertools.combinations(das, 2):
        score = get_ccm(da1, da2)
        if score is None:
            raise ValueError(f"Par nao encontrado na CCM: ({da1}, {da2})")
        scores.append(score)
    return min(scores)


# Gerar todas as configuracoes com w(S)
all_configs = generate_configurations()
for cfg in all_configs:
    cfg['w'] = compute_w(cfg)

print(f"Configuracoes geradas: {len(all_configs)}")
print(f"  w(S) = 1.0: {sum(1 for c in all_configs if c['w'] == 1.0)}")
print(f"  w(S) = 0.5: {sum(1 for c in all_configs if c['w'] == 0.5)}")
print(f"  w(S) = 0.0: {sum(1 for c in all_configs if c['w'] == 0.0)}")

Configuracoes geradas: 144
  w(S) = 1.0: 74
  w(S) = 0.5: 70
  w(S) = 0.0: 0


### Filtragem do conjunto admissivel

Criterio de admissibilidade: w(S) >= 0.5.

CCM tem zero celulas em 0.0 -- todas as configuracoes tem pelo menos compatibilidade condicional em todos os pares.

O threshold w >= 0.5 admite o espaco completo excluindo apenas incompatibilidades absolutas.

In [27]:
def filter_admissible(configs, threshold=0.5):
    """
    Filtra configuracoes com w(S) >= threshold.
    Retorna lista de configuracoes admissiveis com ID sequencial.
    """
    admissible = [c for c in configs if c['w'] >= threshold]
    for i, cfg in enumerate(admissible, 1):
        cfg['id'] = f"S{i:02d}"
    return admissible


admissible = filter_admissible(all_configs, threshold=0.5)

print(f"Espaco admissivel: {len(admissible)} de {len(all_configs)} configuracoes")
print(f"Reducao: {100*(1 - len(admissible)/len(all_configs)):.1f}%")
print()

# Tabela do espaco admissivel
df_admissible = pd.DataFrame([
    {
        'ID': c['id'],
        'D': f"{c['D']} ({DA_NAMES[c['D']]})",
        'I': f"{c['I']} ({DA_NAMES[c['I']]})",
        'ID_': f"{c['ID']} ({DA_NAMES[c['ID']]})",
        'R': f"{c['R']} ({DA_NAMES[c['R']]})",
        'w(S)': c['w']
    }
    for c in admissible
])

print(df_admissible.to_string(index=False))

Espaco admissivel: 144 de 144 configuracoes
Reducao: 0.0%

  ID                     D                        I                     ID_                           R  w(S)
 S01      D1 (model-based)   I1 (output comparison)        ID1 (rule-based)              R1 (exclusion) 1.000
 S02      D1 (model-based)   I1 (output comparison)        ID1 (rule-based) R2 (weight reconfiguration) 1.000
 S03      D1 (model-based)   I1 (output comparison)        ID1 (rule-based)                 R3 (bypass) 1.000
 S04      D1 (model-based)   I1 (output comparison)       ID2 (statistical)              R1 (exclusion) 1.000
 S05      D1 (model-based)   I1 (output comparison)       ID2 (statistical) R2 (weight reconfiguration) 1.000
 S06      D1 (model-based)   I1 (output comparison)       ID2 (statistical)                 R3 (bypass) 0.500
 S07      D1 (model-based)   I1 (output comparison) ID3 (ML identification)              R1 (exclusion) 1.000
 S08      D1 (model-based)   I1 (output comparison) ID3 (ML i

### Resumo do espaco de solucoes


In [28]:
n_total = len(all_configs)
n_admissible = len(admissible)
n_w10 = sum(1 for c in admissible if c['w'] == 1.0)
n_w05 = sum(1 for c in admissible if c['w'] == 0.5)

print("Resumo do espaco de solucoes")
print("-" * 40)
print(f"Espaco bruto:          {n_total} configuracoes (4 x 4 x 3 x 3)")
print(f"Threshold:             w(S) >= 0.5")
print(f"Espaco admissivel:     {n_admissible} configuracoes")
print(f"Reducao:               {100*(1 - n_admissible/n_total):.1f}%")
print(f"  w(S) = 1.0:          {n_w10} configuracoes (sem pares condicionais)")
print(f"  w(S) = 0.5:          {n_w05} configuracoes (ao menos um par condicional)")
print()
print("Interpretacao: CCM nao elimina nenhuma configuracao por")
print("incompatibilidade absoluta. A diferenciacao e feita pelos metodos de selecao.")

Resumo do espaco de solucoes
----------------------------------------
Espaco bruto:          144 configuracoes (4 x 4 x 3 x 3)
Threshold:             w(S) >= 0.5
Espaco admissivel:     144 configuracoes
Reducao:               0.0%
  w(S) = 1.0:          74 configuracoes (sem pares condicionais)
  w(S) = 0.5:          70 configuracoes (ao menos um par condicional)

Interpretacao: CCM nao elimina nenhuma configuracao por
incompatibilidade absoluta. A diferenciacao e feita pelos metodos de selecao.


### Metodos de Selecao

Quatro metodos implementados sobre o espaco admissivel (144 configuracoes).

Cada metodo expoe a mesma interface:
- Input: lista de configuracoes, cenario, Vetor A
- Output: ranking ou conjunto Pareto-eficiente

#### Scoring Linear Ponderado

Formulacao (Levin 2012, Secao 3.5 — Multiple Choice Problem):

$$Q(S) = \sum_{m \in \text{MODULES}} w_m \cdot \sum_{c \in \text{CRITERIA}} w_c \cdot \text{score}(da_m, c)$$

Onde:
- $w_m$: peso do modulo $m$ no cenario (homogeneo = 0.25)
- $w_c$: peso do criterio $c$ no cenario
- $\text{score}(da_m, c)$: score da DA selecionada para o modulo $m$ no criterio $c$ (Vetor A)

Output: escalar $Q(S)$ por configuracao — ranking por valor decrescente.

In [29]:
def scoring_linear(admissible, scenario, vetor_a):
    w_mod = scenario['module_weights']
    w_crit = scenario['criteria_weights']
    results = []

    for cfg in admissible:
        q = 0.0
        for mod in MODULES:
            da = cfg[mod]
            da_score = sum(
                w_crit[c] * vetor_a.loc[da, c]
                for c in CRITERIA
            )
            q += w_mod[mod] * da_score
        results.append({'id': cfg['id'], 'Q': round(q, 4)})

    results.sort(key=lambda x: x['Q'], reverse=True)
    for rank, r in enumerate(results, 1):
        r['rank'] = rank
    return results


# Executar para todos os cenarios
print("Scoring Linear Ponderado -- Top 10 por cenario")
print("=" * 60)

scoring_results = {}
for sc_id, sc in SCENARIOS.items():
    res = scoring_linear(admissible, sc, vetor_a)
    scoring_results[sc_id] = res
    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'Rank':<6} {'ID':<6} {'Q':>6}  {'D':<6} {'I':<6} {'ID':<6} {'R':<6}")
    print("-" * 55)
    cfg_map = {c['id']: c for c in admissible}
    for r in res[:10]:
        cfg = cfg_map[r['id']]
        print(f"{r['rank']:<6} {r['id']:<6} {r['Q']:>6.4f}  "
              f"{cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<6}")

Scoring Linear Ponderado -- Top 10 por cenario

SC1 -- Uniforme
Rank   ID          Q  D      I      ID     R     
-------------------------------------------------------
1      S01    4.4000  D1     I1     ID1    R1    
2      S37    4.4000  D2     I1     ID1    R1    
3      S03    4.3500  D1     I1     ID1    R3    
4      S10    4.3500  D1     I2     ID1    R1    
5      S39    4.3500  D2     I1     ID1    R3    
6      S46    4.3500  D2     I2     ID1    R1    
7      S12    4.3000  D1     I2     ID1    R3    
8      S48    4.3000  D2     I2     ID1    R3    
9      S28    4.2500  D1     I4     ID1    R1    
10     S64    4.2500  D2     I4     ID1    R1    

SC2 -- Segurança / certificação
Rank   ID          Q  D      I      ID     R     
-------------------------------------------------------
1      S01    4.7000  D1     I1     ID1    R1    
2      S10    4.6750  D1     I2     ID1    R1    
3      S37    4.6375  D2     I1     ID1    R1    
4      S03    4.6125  D1     I1     ID1  

#### Metodo do Ponto Ideal

Calcula a distancia de cada configuracao ao ponto ideal S^o.

$S^o$: para cada modulo, a DA com maior score ponderado pelos criterios.

$$S^* = \arg\min_{S \in \{S_a\}} \rho(S, S^o)$$

$$\rho(S, S^o) = \sqrt{\sum_{m \in \text{MODULES}} w_m \cdot \sum_{c \in \text{CRITERIA}} w_c \cdot (score^o_c - score(da_m, c))^2}$$

Onde $S^o$ e a configuracao ideal formada pela melhor DA de cada modulo
por criterio ponderado. Output: distancia $\rho$ por configuracao --
ranking por valor crescente (menor distancia = melhor).

In [30]:

def ideal_point(admissible, scenario, vetor_a):
    w_mod = scenario['module_weights']
    w_crit = scenario['criteria_weights']

    # Calcular ponto ideal: melhor DA por modulo (maior score ponderado)
    ideal = {}
    for mod in MODULES:
        best_da = max(
            MODULE_VALUES[mod],
            key=lambda da: sum(w_crit[c] * vetor_a.loc[da, c] for c in CRITERIA)
        )
        ideal[mod] = {c: vetor_a.loc[best_da, c] for c in CRITERIA}

    results = []
    for cfg in admissible:
        rho_sq = 0.0
        for mod in MODULES:
            da = cfg[mod]
            for c in CRITERIA:
                diff = ideal[mod][c] - vetor_a.loc[da, c]
                rho_sq += w_mod[mod] * w_crit[c] * (diff ** 2)
        rho = round(rho_sq ** 0.5, 4)
        results.append({'id': cfg['id'], 'rho': rho})

    results.sort(key=lambda x: x['rho'])
    for rank, r in enumerate(results, 1):
        r['rank'] = rank
    return results


# Executar para todos os cenarios
print("Ideal Point -- Top 10 por cenario")
print("=" * 60)

ideal_results = {}
for sc_id, sc in SCENARIOS.items():
    res = ideal_point(admissible, sc, vetor_a)
    ideal_results[sc_id] = res
    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'Rank':<6} {'ID':<6} {'rho':>7}  {'D':<6} {'I':<6} {'ID':<6} {'R':<6}")
    print("-" * 55)
    cfg_map = {c['id']: c for c in admissible}
    for r in res[:10]:
        cfg = cfg_map[r['id']]
        print(f"{r['rank']:<6} {r['id']:<6} {r['rho']:>7.4f}  "
              f"{cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<6}")

Ideal Point -- Top 10 por cenario

SC1 -- Uniforme
Rank   ID         rho  D      I      ID     R     
-------------------------------------------------------
1      S01     0.0000  D1     I1     ID1    R1    
2      S10     0.2236  D1     I2     ID1    R1    
3      S03     0.3873  D1     I1     ID1    R3    
4      S12     0.4472  D1     I2     ID1    R3    
5      S37     0.4472  D2     I1     ID1    R1    
6      S28     0.5000  D1     I4     ID1    R1    
7      S46     0.5000  D2     I2     ID1    R1    
8      S39     0.5916  D2     I1     ID1    R3    
9      S30     0.6325  D1     I4     ID1    R3    
10     S48     0.6325  D2     I2     ID1    R3    

SC2 -- Segurança / certificação
Rank   ID         rho  D      I      ID     R     
-------------------------------------------------------
1      S01     0.0000  D1     I1     ID1    R1    
2      S10     0.1581  D1     I2     ID1    R1    
3      S03     0.3708  D1     I1     ID1    R3    
4      S12     0.4031  D1     I2     ID

#### Pareto-Based Morphological Analysis

Uma configuracao $S$ e Pareto-eficiente se nao existe $S'$ tal que:

$$\forall c \in \text{CRITERIA}: f_c(S') \geq f_c(S) \quad \text{e} \quad \exists c: f_c(S') > f_c(S)$$

Onde $f_c(S)$ e o score agregado da configuracao $S$ no criterio $c$:

$$f_c(S) = \sum_{m \in \text{MODULES}} w_m \cdot \text{score}(da_m, c)$$

Output: conjunto de configuracoes nao-dominadas (front Pareto).
O Pareto-based MA nao depende de cenario para determinar dominancia --
os pesos dos modulos entram apenas no calculo de $f_c(S)$.

In [31]:
def compute_criteria_scores(admissible, scenario, vetor_a):
    w_mod = scenario['module_weights']
    scores = {}
    for cfg in admissible:
        scores[cfg['id']] = {
            c: sum(w_mod[mod] * vetor_a.loc[cfg[mod], c] for mod in MODULES)
            for c in CRITERIA
        }
    return scores


def pareto_based_ma(admissible, scenario, vetor_a):
    crit_scores = compute_criteria_scores(admissible, scenario, vetor_a)
    ids = [cfg['id'] for cfg in admissible]
    dominated = set()

    for i, s1 in enumerate(ids):
        for s2 in ids:
            if s1 == s2:
                continue
            # Verificar se s1 e dominada por s2
            geq_all = all(crit_scores[s2][c] >= crit_scores[s1][c] for c in CRITERIA)
            gt_one  = any(crit_scores[s2][c] >  crit_scores[s1][c] for c in CRITERIA)
            if geq_all and gt_one:
                dominated.add(s1)
                break

    pareto_front = [cfg for cfg in admissible if cfg['id'] not in dominated]
    return pareto_front, crit_scores


# Executar para todos os cenarios
print("Pareto-Based MA -- Front Pareto por cenario")
print("=" * 60)

pareto_results = {}
for sc_id, sc in SCENARIOS.items():
    front, crit_scores = pareto_based_ma(admissible, sc, vetor_a)
    pareto_results[sc_id] = {'front': front, 'crit_scores': crit_scores}

    print(f"\n{sc_id} -- {sc['name']}")
    print(f"Front Pareto: {len(front)} de {len(admissible)} configuracoes")
    print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<6} "
          f"{'f_C1':>6} {'f_C2':>6} {'f_C3':>6} {'f_C4':>6} {'f_C5':>6}")
    print("-" * 70)
    for cfg in sorted(front, key=lambda x: x['id']):
        cs = crit_scores[cfg['id']]
        print(f"{cfg['id']:<6} {cfg['D']:<6} {cfg['I']:<6} "
              f"{cfg['ID']:<6} {cfg['R']:<6} "
              f"{cs['C1']:>6.3f} {cs['C2']:>6.3f} {cs['C3']:>6.3f} "
              f"{cs['C4']:>6.3f} {cs['C5']:>6.3f}")

Pareto-Based MA -- Front Pareto por cenario

SC1 -- Uniforme
Front Pareto: 6 de 144 configuracoes
ID     D      I      ID     R        f_C1   f_C2   f_C3   f_C4   f_C5
----------------------------------------------------------------------
S01    D1     I1     ID1    R1      5.000  4.000  4.500  5.000  3.500
S03    D1     I1     ID1    R3      5.000  3.750  4.500  4.750  3.750
S04    D1     I1     ID2    R1      4.500  3.500  4.250  5.000  3.750
S06    D1     I1     ID2    R3      4.500  3.250  4.250  4.750  4.000
S37    D2     I1     ID1    R1      4.750  4.250  4.750  5.000  3.250
S39    D2     I1     ID1    R3      4.750  4.000  4.750  4.750  3.500

SC2 -- Segurança / certificação
Front Pareto: 6 de 144 configuracoes
ID     D      I      ID     R        f_C1   f_C2   f_C3   f_C4   f_C5
----------------------------------------------------------------------
S01    D1     I1     ID1    R1      5.000  4.000  4.500  5.000  3.500
S03    D1     I1     ID1    R3      5.000  3.750  4.500  4.7

#### Hierarchical Morphological Multicriteria Design (HMMD)

O vetor de qualidade do sistema e definido como:

$$N(S) = (w(S);\, e(S))$$

Onde:
- $w(S) = \min_{j_1 \neq j_2} \text{CCM}(da_{j_1}, da_{j_2})$ — minimo das compatibilidades par-a-par
- $e(S) = (\eta_1, \eta_2, \eta_3, \eta_4)$ — vetor multiset contando DAs por nivel de prioridade, com $\sum_r \eta_r = m$ (numero de modulos)

**Prioridades ordinais das DAs por modulo:**
Calculadas por score ponderado pelos criterios do cenario (D07):

$$\text{score}(da, sc) = \sum_{c \in \text{CRITERIA}} w_c(sc) \cdot \text{score}(da, c)$$

A posicao ordinal dentro do modulo e determinada pelo ranking desse score
(posicao 1 = maior score). Empates recebem a mesma posicao (D05).
Dimensao $k = 4$ global para todos os modulos (D06).

**Criterio de Pareto-eficiencia sobre $N(S)$:**
$S$ e nao-dominada se nao existe $S'$ tal que $N(S') \succ N(S)$,
onde $\succ$ denota dominancia lexicografica sobre $(w(S); e(S))$.

**Problema de otimizacao (Levin 2015, Secao 2.2.7):**

$$\max\, e(S), \quad \max\, w(S), \quad \text{s.t.} \quad w(S) \geq 0.5$$

In [32]:
def compute_priorities(scenario, vetor_a):
    w_crit = scenario['criteria_weights']
    priorities = {}

    for mod in MODULES:
        das = MODULE_VALUES[mod]
        scores = {
            da: sum(w_crit[c] * vetor_a.loc[da, c] for c in CRITERIA)
            for da in das
        }
        # Ranking com empates na mesma posicao (dense ranking)
        sorted_scores = sorted(set(scores.values()), reverse=True)
        rank_map = {s: i+1 for i, s in enumerate(sorted_scores)}
        priorities[mod] = {da: rank_map[scores[da]] for da in das}

    return priorities


def compute_e(config, priorities, k=4):
    e = [0] * k
    for mod in MODULES:
        da = config[mod]
        rank = priorities[mod][da]
        if rank <= k:
            e[rank - 1] += 1
    return tuple(e)


def hmmd_dominates(n1, n2):
    w1, e1 = n1
    w2, e2 = n2

    # Comparacao de w
    w_geq = w1 >= w2
    w_gt  = w1 > w2

    # Comparacao de e pelo poset (lexicografico)
    e_geq = e1 >= e2  # tuple comparison -- lexicografico
    e_gt  = e1 > e2

    return (w_geq and e_geq) and (w_gt or e_gt)


def hmmd(admissible, scenario, vetor_a, k=4):
    priorities = compute_priorities(scenario, vetor_a)

    # Calcular N(S) para cada configuracao
    ns = {}
    for cfg in admissible:
        w = cfg['w']
        e = compute_e(cfg, priorities, k)
        ns[cfg['id']] = (w, e)

    # Identificar solucoes nao-dominadas
    dominated = set()
    ids = [cfg['id'] for cfg in admissible]
    for s1 in ids:
        for s2 in ids:
            if s1 == s2:
                continue
            if hmmd_dominates(ns[s2], ns[s1]):
                dominated.add(s1)
                break

    pareto_hmmd = [cfg for cfg in admissible if cfg['id'] not in dominated]
    return pareto_hmmd, ns, priorities


# Executar para todos os cenarios
print("HMMD -- Front Pareto por cenario")
print("=" * 60)

hmmd_results = {}
for sc_id, sc in SCENARIOS.items():
    front, ns, priorities = hmmd(admissible, sc, vetor_a)
    hmmd_results[sc_id] = {'front': front, 'ns': ns, 'priorities': priorities}

    print(f"\n{sc_id} -- {sc['name']}")
    print(f"Front HMMD: {len(front)} de {len(admissible)} configuracoes")
    print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<6} "
          f"{'w(S)':>6} {'e(S)':<20} {'N(S)'}")
    print("-" * 75)
    for cfg in sorted(front, key=lambda x: x['id']):
        n = ns[cfg['id']]
        print(f"{cfg['id']:<6} {cfg['D']:<6} {cfg['I']:<6} "
              f"{cfg['ID']:<6} {cfg['R']:<6} "
              f"{n[0]:>6} {str(n[1]):<20} ({n[0]}, {n[1]})")

HMMD -- Front Pareto por cenario

SC1 -- Uniforme
Front HMMD: 2 de 144 configuracoes
ID     D      I      ID     R        w(S) e(S)                 N(S)
---------------------------------------------------------------------------
S01    D1     I1     ID1    R1        1.0 (4, 0, 0, 0)         (1.0, (4, 0, 0, 0))
S37    D2     I1     ID1    R1        1.0 (4, 0, 0, 0)         (1.0, (4, 0, 0, 0))

SC2 -- Segurança / certificação
Front HMMD: 1 de 144 configuracoes
ID     D      I      ID     R        w(S) e(S)                 N(S)
---------------------------------------------------------------------------
S01    D1     I1     ID1    R1        1.0 (4, 0, 0, 0)         (1.0, (4, 0, 0, 0))

SC3 -- Cobertura / desempenho
Front HMMD: 1 de 144 configuracoes
ID     D      I      ID     R        w(S) e(S)                 N(S)
---------------------------------------------------------------------------
S03    D1     I1     ID1    R3        1.0 (4, 0, 0, 0)         (1.0, (4, 0, 0, 0))

SC4 -- Eficiênci

## Experimentos Consolidados

Execução dos 4 métodos de seleção sobre o espaço admissível (144 configurações).
Inputs: Vetor A escala 1-4 (consolidado de 4 LLMs, pós-revisão) · CCM escala {0,1,2,3}.

**Métodos:**
- Scoring linear ponderado: ranking por Q(S) decrescente
- Ideal point: ranking por distância ρ crescente ao ponto ideal
- Pareto-based MA: conjunto não-dominado sobre vetores de critérios
- HMMD: conjunto não-dominado sobre N(S) = (w(S); e(S))

In [33]:
# Mapa de configuracoes para lookup rapido
cfg_map = {c['id']: c for c in admissible}

# 4.1 Tabela comparativa -- Scoring e Ideal Point por cenario
print("Secao 4 -- Comparacao de Rankings")
print("=" * 60)

for sc_id, sc in SCENARIOS.items():
    s_res = scoring_results[sc_id]
    i_res = ideal_results[sc_id]

    # Converter para dict rank por ID
    s_rank = {r['id']: r['rank'] for r in s_res}
    i_rank = {r['id']: r['rank'] for r in i_res}

    # Top 10 uniao (IDs que aparecem em qualquer top 10)
    top_ids = list(dict.fromkeys(
        [r['id'] for r in s_res[:10]] +
        [r['id'] for r in i_res[:10]]
    ))

    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<4} "
          f"{'Scoring':>8} {'Q':>7} {'Ideal':>7} {'rho':>7} {'Pareto':>8} {'HMMD':>6}")
    print("-" * 72)

    # Pareto front IDs para este cenario
    pareto_ids = {c['id'] for c in pareto_results[sc_id]['front']}
    hmmd_ids = {c['id'] for c in hmmd_results[sc_id]['front']}

    s_dict = {r['id']: r for r in s_res}
    i_dict = {r['id']: r for r in i_res}

    for sid in top_ids:
        cfg = cfg_map[sid]
        s_r = s_rank.get(sid, '-')
        i_r = i_rank.get(sid, '-')
        q = s_dict[sid]['Q']
        rho = i_dict[sid]['rho']
        pareto = 'Pareto' if sid in pareto_ids else ''
        hmmd_flag = 'HMMD' if sid in hmmd_ids else ''
        print(f"{sid:<6} {cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<4} "
              f"{s_r:>8} {q:>7.4f} {i_r:>7} {rho:>7.4f} {pareto:>8} {hmmd_flag:>6}")

Secao 4 -- Comparacao de Rankings

SC1 -- Uniforme
ID     D      I      ID     R     Scoring       Q   Ideal     rho   Pareto   HMMD
------------------------------------------------------------------------
S01    D1     I1     ID1    R1          1  4.4000       1  0.0000   Pareto   HMMD
S37    D2     I1     ID1    R1          2  4.4000       5  0.4472   Pareto   HMMD
S03    D1     I1     ID1    R3          3  4.3500       3  0.3873   Pareto       
S10    D1     I2     ID1    R1          4  4.3500       2  0.2236                
S39    D2     I1     ID1    R3          5  4.3500       8  0.5916   Pareto       
S46    D2     I2     ID1    R1          6  4.3500       7  0.5000                
S12    D1     I2     ID1    R3          7  4.3000       4  0.4472                
S48    D2     I2     ID1    R3          8  4.3000      10  0.6325                
S28    D1     I4     ID1    R1          9  4.2500       6  0.5000                
S64    D2     I4     ID1    R1         10  4.2500      1

## Analise Comparativa

Correlacao de Spearman entre metodos, identificacao de divergencias
e achados transversais.

In [34]:
from scipy.stats import spearmanr

print("Secao 5 -- Analise Comparativa")
print("=" * 60)

# 5.1 Correlacao de Spearman -- Scoring vs Ideal Point
print("\n5.1 Correlacao de Spearman -- Scoring vs Ideal Point")
print("-" * 50)
print(f"{'Cenario':<8} {'rho':>8} {'p':>10}")
print("-" * 30)

for sc_id in SCENARIOS:
    s_res = scoring_results[sc_id]
    i_res = ideal_results[sc_id]

    # Alinhar rankings pelo ID
    ids = [r['id'] for r in s_res]
    s_ranks = [r['rank'] for r in s_res]
    i_rank_map = {r['id']: r['rank'] for r in i_res}
    i_ranks = [i_rank_map[sid] for sid in ids]

    rho, p = spearmanr(s_ranks, i_ranks)
    print(f"{sc_id:<8} {rho:>8.4f} {p:>10.4f}")

# 5.2 Correlacao de Spearman -- entre cenarios (scoring)
print("\n5.2 Correlacao de Spearman -- Scoring entre cenarios")
print("-" * 50)
sc_ids = list(SCENARIOS.keys())
print(f"{'':>8}", end='')
for sc in sc_ids:
    print(f"{sc:>8}", end='')
print()

for sc1 in sc_ids:
    print(f"{sc1:<8}", end='')
    r1 = {r['id']: r['rank'] for r in scoring_results[sc1]}
    for sc2 in sc_ids:
        r2 = {r['id']: r['rank'] for r in scoring_results[sc2]}
        ids = list(r1.keys())
        rho, _ = spearmanr([r1[i] for i in ids], [r2[i] for i in ids])
        print(f"{rho:>8.4f}", end='')
    print()

# 5.3 Divergencias entre metodos
print("\n5.3 Divergencias identificadas")
print("-" * 50)

for sc_id, sc in SCENARIOS.items():
    s_top10 = {r['id'] for r in scoring_results[sc_id][:10]}
    i_top10 = {r['id'] for r in ideal_results[sc_id][:10]}
    pareto_ids = {c['id'] for c in pareto_results[sc_id]['front']}

    # Em Pareto mas nao no top 10 scoring
    pareto_not_scoring = pareto_ids - s_top10
    # No top 10 scoring mas nao no Pareto
    scoring_not_pareto = s_top10 - pareto_ids

    print(f"\n{sc_id} -- {sc['name']}")
    if pareto_not_scoring:
        print(f"  Pareto-eficiente mas fora do top 10 scoring: "
              f"{', '.join(sorted(pareto_not_scoring))}")
        for sid in sorted(pareto_not_scoring):
            cfg = cfg_map[sid]
            cs = pareto_results[sc_id]['crit_scores'][sid]
            print(f"    {sid}: D={cfg['D']} I={cfg['I']} "
                  f"ID={cfg['ID']} R={cfg['R']}")
            print(f"    vetores C1-C5: "
                  f"{[round(float(cs[c]),3) for c in CRITERIA]}")
    if scoring_not_pareto:
        print(f"  Top 10 scoring mas nao Pareto-eficiente: "
              f"{', '.join(sorted(scoring_not_pareto))}")
    if not pareto_not_scoring and not scoring_not_pareto:
        print("  Sem divergencias relevantes.")

# 5.4 Achados transversais
print("\n5.4 Achados transversais")
print("-" * 50)

# Estabilidade do ranking
rho_min = 1.0
rho_min_pair = ('', '')
for i, sc1 in enumerate(sc_ids):
    for sc2 in sc_ids[i+1:]:
        r1 = {r['id']: r['rank'] for r in scoring_results[sc1]}
        r2 = {r['id']: r['rank'] for r in scoring_results[sc2]}
        ids = list(r1.keys())
        rho, _ = spearmanr([r1[i] for i in ids], [r2[i] for i in ids])
        if rho < rho_min:
            rho_min = rho
            rho_min_pair = (sc1, sc2)

print(f"Spearman minimo entre cenarios (scoring): "
      f"{rho_min:.4f} ({rho_min_pair[0]} x {rho_min_pair[1]})")

# Calcular rank medio e maximo por configuracao
rank_stats = {}
for sid in [c['id'] for c in admissible]:
    ranks = [r['rank'] for sc in sc_ids
             for r in scoring_results[sc] if r['id'] == sid]
    rank_stats[sid] = {
        'mean': sum(ranks) / len(ranks),
        'max': max(ranks),
        'ranks': ranks
    }

# Ordenar por rank medio, desempate por rank maximo
sorted_by_robustness = sorted(
    rank_stats.items(),
    key=lambda x: (x[1]['mean'], x[1]['max'])
)

# Pareto e HMMD em todos os cenarios
pareto_todos = set.intersection(*[
    {c['id'] for c in pareto_results[sc]['front']}
    for sc in sc_ids
])
hmmd_todos = set.intersection(*[
    {c['id'] for c in hmmd_results[sc]['front']}
    for sc in sc_ids
])

print("Top 5 configuracoes por robustez (rank medio no scoring):")
print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<6} "
      f"{'Ranks SC1-SC4':<20} {'Media':>6} {'Max':>5} {'Pareto':>8} {'HMMD':>6}")
print("-" * 80)
for sid, stats in sorted_by_robustness[:5]:
    cfg = cfg_map[sid]
    pareto_flag = 'Pareto' if sid in pareto_todos else ''
    hmmd_flag = 'HMMD' if sid in hmmd_todos else ''
    print(f"{sid:<6} {cfg['D']:<6} {cfg['I']:<6} {cfg['ID']:<6} {cfg['R']:<6} "
          f"{str(stats['ranks']):<20} {stats['mean']:>6.2f} {stats['max']:>5} "
          f"{pareto_flag:>8} {hmmd_flag:>6}")

# Trade-off identificado
print(f"\nTrade-off C5 vs C1-C4:")
print(f"  S50 (D2, I2, ID2, R2) -- Pareto-eficiente em todos os cenarios")
cs_s50 = pareto_results['SC1']['crit_scores']['S50']
print(f"  Vetor C1-C5: {[round(float(cs_s50[c]), 3) for c in CRITERIA]}")
print(f"  C5 maior que S46/S48, mas C1-C4 menores.")
print(f"  Nao aparece no top 10 do scoring em nenhum cenario.")

Secao 5 -- Analise Comparativa

5.1 Correlacao de Spearman -- Scoring vs Ideal Point
--------------------------------------------------
Cenario       rho          p
------------------------------
SC1        0.9680     0.0000
SC2        0.9714     0.0000
SC3        0.9421     0.0000
SC4        0.9706     0.0000

5.2 Correlacao de Spearman -- Scoring entre cenarios
--------------------------------------------------
             SC1     SC2     SC3     SC4
SC1       1.0000  0.9904  0.9785  0.9767
SC2       0.9904  1.0000  0.9524  0.9523
SC3       0.9785  0.9524  1.0000  0.9461
SC4       0.9767  0.9523  0.9461  1.0000

5.3 Divergencias identificadas
--------------------------------------------------

SC1 -- Uniforme
  Pareto-eficiente mas fora do top 10 scoring: S04, S06
    S04: D=D1 I=I1 ID=ID2 R=R1
    vetores C1-C5: [4.5, 3.5, 4.25, 5.0, 3.75]
    S06: D=D1 I=I1 ID=ID2 R=R3
    vetores C1-C5: [4.5, 3.25, 4.25, 4.75, 4.0]
  Top 10 scoring mas nao Pareto-eficiente: S10, S12, S28, S46, S4

In [35]:
def compute_priorities(scenario, vetor_a):
    w_crit = scenario['criteria_weights']
    priorities = {}

    for mod in MODULES:
        das = MODULE_VALUES[mod]
        scores = {
            da: sum(w_crit[c] * vetor_a.loc[da, c] for c in CRITERIA)
            for da in das
        }
        # Ranking com empates na mesma posicao (dense ranking)
        sorted_scores = sorted(set(scores.values()), reverse=True)
        rank_map = {s: i+1 for i, s in enumerate(sorted_scores)}
        priorities[mod] = {da: rank_map[scores[da]] for da in das}

    return priorities


def compute_e(config, priorities, k=4):
    e = [0] * k
    for mod in MODULES:
        da = config[mod]
        rank = priorities[mod][da]
        if rank <= k:
            e[rank - 1] += 1
    return tuple(e)


def hmmd_dominates(n1, n2):
    w1, e1 = n1
    w2, e2 = n2

    # Comparacao de w
    w_geq = w1 >= w2
    w_gt  = w1 > w2

    # Comparacao de e pelo poset (lexicografico)
    e_geq = e1 >= e2  # tuple comparison -- lexicografico
    e_gt  = e1 > e2

    return (w_geq and e_geq) and (w_gt or e_gt)


def hmmd(admissible, scenario, vetor_a, k=4):
    priorities = compute_priorities(scenario, vetor_a)

    # Calcular N(S) para cada configuracao
    ns = {}
    for cfg in admissible:
        w = cfg['w']
        e = compute_e(cfg, priorities, k)
        ns[cfg['id']] = (w, e)

    # Identificar solucoes nao-dominadas
    dominated = set()
    ids = [cfg['id'] for cfg in admissible]
    for s1 in ids:
        for s2 in ids:
            if s1 == s2:
                continue
            if hmmd_dominates(ns[s2], ns[s1]):
                dominated.add(s1)
                break

    pareto_hmmd = [cfg for cfg in admissible if cfg['id'] not in dominated]
    return pareto_hmmd, ns, priorities


# Executar para todos os cenarios
print("HMMD -- Front Pareto por cenario")
print("=" * 60)

hmmd_results = {}
for sc_id, sc in SCENARIOS.items():
    front, ns, priorities = hmmd(admissible, sc, vetor_a)
    hmmd_results[sc_id] = {'front': front, 'ns': ns, 'priorities': priorities}

    print(f"\n{sc_id} -- {sc['name']}")
    print(f"Front HMMD: {len(front)} de {len(admissible)} configuracoes")
    print(f"{'ID':<6} {'D':<6} {'I':<6} {'ID':<6} {'R':<6} "
          f"{'w(S)':>6} {'e(S)':<20} {'N(S)'}")
    print("-" * 75)
    for cfg in sorted(front, key=lambda x: x['id']):
        n = ns[cfg['id']]
        print(f"{cfg['id']:<6} {cfg['D']:<6} {cfg['I']:<6} "
              f"{cfg['ID']:<6} {cfg['R']:<6} "
              f"{n[0]:>6.1f} {str(n[1]):<20} ({n[0]}, {n[1]})")

HMMD -- Front Pareto por cenario

SC1 -- Uniforme
Front HMMD: 2 de 144 configuracoes
ID     D      I      ID     R        w(S) e(S)                 N(S)
---------------------------------------------------------------------------
S01    D1     I1     ID1    R1        1.0 (4, 0, 0, 0)         (1.0, (4, 0, 0, 0))
S37    D2     I1     ID1    R1        1.0 (4, 0, 0, 0)         (1.0, (4, 0, 0, 0))

SC2 -- Segurança / certificação
Front HMMD: 1 de 144 configuracoes
ID     D      I      ID     R        w(S) e(S)                 N(S)
---------------------------------------------------------------------------
S01    D1     I1     ID1    R1        1.0 (4, 0, 0, 0)         (1.0, (4, 0, 0, 0))

SC3 -- Cobertura / desempenho
Front HMMD: 1 de 144 configuracoes
ID     D      I      ID     R        w(S) e(S)                 N(S)
---------------------------------------------------------------------------
S03    D1     I1     ID1    R3        1.0 (4, 0, 0, 0)         (1.0, (4, 0, 0, 0))

SC4 -- Eficiênci

In [36]:
# Diagnostico de prioridades por cenario e modulo
for sc_id, sc in SCENARIOS.items():
    w_crit = sc['criteria_weights']
    print(f"\n{sc_id} -- {sc['name']}")
    print(f"{'DA':<6} {'Modulo':<8} {'Score':>7} {'Rank':>6}")
    print("-" * 35)
    for mod in MODULES:
        das = MODULE_VALUES[mod]
        scores = {
            da: round(sum(w_crit[c] * vetor_a.loc[da, c] for c in CRITERIA), 4)
            for da in das
        }
        sorted_scores = sorted(set(scores.values()), reverse=True)
        rank_map = {s: i+1 for i, s in enumerate(sorted_scores)}
        for da in das:
            print(f"{da:<6} {mod:<8} {scores[da]:>7.4f} {rank_map[scores[da]]:>6}")


SC1 -- Uniforme
DA     Modulo     Score   Rank
-----------------------------------
D1     D         4.2000      1
D2     D         4.2000      1
D3     D         2.2000      3
D4     D         2.8000      2
I1     I         4.2000      1
I2     I         4.0000      2
I3     I         3.0000      4
I4     I         3.6000      3
ID1    ID        4.6000      1
ID2    ID        3.8000      2
ID3    ID        2.2000      3
R1     R         4.6000      1
R2     R         3.6000      3
R3     R         4.4000      2

SC2 -- Segurança / certificação
DA     Modulo     Score   Rank
-----------------------------------
D1     D         4.6000      1
D2     D         4.3500      2
D3     D         2.1000      4
D4     D         2.9000      3
I1     I         4.6000      1
I2     I         4.5000      2
I3     I         3.2500      4
I4     I         3.5500      3
ID1    ID        4.8000      1
ID2    ID        3.9000      2
ID3    ID        2.1000      3
R1     R         4.8000      1
R2     R  

---
## Referencias

Bader, K., Lussier, B., & Schön, W. (2017a). A fault tolerant architecture
for data fusion: A real application of Kalman filters for mobile robot
localization. *Robotics and Autonomous Systems*, *88*, 11-23.
https://doi.org/10.1016/j.robot.2016.12.001

Bader, K., Lussier, B., & Schön, W. (2017b). Fault tolerance from formal
analysis of a data fusion mechanism. In *2017 First IEEE International
Conference on Robotic Computing (IRC)* (pp. 69-74). IEEE.
https://doi.org/10.1109/IRC.2017.58

Ceccarelli, A., & Secci, F. (2023). RGB cameras failures and their effects
in autonomous driving applications. *IEEE Transactions on Dependable and
Secure Computing*, *20*(4), 2731-2745.
https://doi.org/10.1109/TDSC.2022.3156941

Conejo, C., Puig, V., Morcego, B., Navas, F., & Milanes, V. (2025).
Enhancing safety in autonomous vehicles using zonotopic LPV-EKF for fault
detection and isolation in state estimation. *Control Engineering Practice*,
*156*, 106192. https://doi.org/10.1016/j.conengprac.2024.106192

Elhoseny, M., Rao, D. D., Veerasamy, B. D., Alduaiji, N., Shreyas, J., &
Shukla, P. K. (2024). Deep learning algorithm for optimized sensor data
fusion in fault diagnosis and tolerance. *International Journal of
Computational Intelligence Systems*, *17*, 299.
https://doi.org/10.1007/s44196-024-00692-5

Emzivat, Y., Delahaye, B., Roux, O. H., & El Najjar, M. E. (2018). A formal
approach for the design of a dependable perception system for autonomous
vehicles. In *2018 IEEE Intelligent Vehicles Symposium (IV)* (pp. 1297-1304).
IEEE. https://doi.org/10.1109/IVS.2018.8500412

Erhan, L., Ndubuaku, M., Di Mauro, M., Song, W., Chen, M., Fortino, G.,
Bagdasar, O., & Liotta, A. (2021). Smart anomaly detection in sensor systems:
A multi-perspective review. *Information Fusion*, *67*, 64-79.
https://doi.org/10.1016/j.inffus.2020.10.001

Gao, Z., Cecati, C., & Ding, S. X. (2015a). A survey of fault diagnosis and
fault-tolerant techniques -- Part I: Fault diagnosis with model-based and
signal-based approaches. *IEEE Transactions on Industrial Electronics*,
*62*(6), 3757-3767. https://doi.org/10.1109/TIE.2015.2417570

Gao, Z., Cecati, C., & Ding, S. X. (2015b). A survey of fault diagnosis and
fault-tolerant techniques -- Part II: Fault diagnosis with knowledge-based
and hybrid/active approaches. *IEEE Transactions on Industrial Electronics*,
*62*(6), 3768-3774. https://doi.org/10.1109/TIE.2015.2419013

Goelles, T., Schlager, B., & Muckenhuber, S. (2020). Fault detection,
isolation, identification and recovery (FDIIR) methods for automotive
perception sensors including a detailed literature survey for lidar.
*Sensors*, *20*(13), 3662. https://doi.org/10.3390/s20133662

Grubmuller, S., Stettinger, G., Sotelo, M. A., & Watzenig, D. (2019).
Fault-tolerant environmental perception architecture for robust automated
driving. In *2019 IEEE International Conference on Connected Vehicles and
Expo (ICCVE)* (pp. 1-6). IEEE.
https://doi.org/10.1109/ICCVE45908.2019.8965112

Hao, Y., Ding, Y., Wang, G., Zhou, Y., & Jia, X. (2019). A fault-tolerant
data fusion method based on decentralized Kalman filter for redundant sensor
configuration. In *Proceedings of the 38th Chinese Control Conference*
(pp. 4486-4491). IEEE. https://doi.org/10.23919/ChiCC.2019.8856985

Hou, W., Li, W., & Li, P. (2023). Fault diagnosis of the autonomous driving
perception system based on information fusion. *Sensors*, *23*(11), 5110.
https://doi.org/10.3390/s23115110

Koopman, P., & Wagner, M. (2017). Autonomous vehicle safety: An
interdisciplinary challenge. *IEEE Intelligent Transportation Systems
Magazine*, *9*(1), 90-96. https://doi.org/10.1109/MITS.2016.2583491

Lee, J., Kang, H., Jeon, Y., Cho, J., Kim, S., & Jo, K. (2024). Sensor
fusion-based classification fault-tolerant system for detected objects in
autonomous cars. *IEEE Transactions on Intelligent Vehicles*.
https://doi.org/10.1109/TIV.2024.3361001

Mendonca, R., Rodrigues, J., Rodrigues, R., & Mendonca, R. (2023). Enhancing
the reliability of perception systems using N-version programming and
rejuvenation. In *2023 IEEE 28th Pacific Rim International Symposium on
Dependable Computing (PRDC)* (pp. 11-20). IEEE.
https://doi.org/10.1109/PRDC59308.2023.00012

Min, H., Fang, Y., Wu, X., Lei, X., Chen, S., Teixeira, R., Zhu, B., Zhao,
X., & Xu, Z. (2023). A fault diagnosis framework for autonomous vehicles with
sensor self-diagnosis. *Expert Systems with Applications*, *224*, 120002.
https://doi.org/10.1016/j.eswa.2023.120002

Nitsch, J., Itkina, M., Senanayake, R., Nieto, J., Schmidt, M., Siegwart,
R., Kochenderfer, M. J., & Cadena, C. (2021). Out-of-distribution detection
for automotive perception. In *2021 IEEE International Intelligent
Transportation Systems Conference (ITSC)* (pp. 2938-2943). IEEE.
https://doi.org/10.1109/ITSC48978.2021.9564545

Sinha, R., Schmerling, E., & Pavone, M. (2023). Closing the loop on runtime
monitors with fallback-safe MPC. *arXiv preprint arXiv:2309.08603*.
https://doi.org/10.48550/arXiv.2309.08603

Tian, H., Ding, W., Han, X., Wu, G., Guo, A., Zhang, J., Chen, W., Wei, J.,
& Zhang, T. (2025). Testing the fault-tolerance of multi-sensor fusion
perception in autonomous driving systems. *Proceedings of the ACM on
Software Engineering*, *2*(ISSTA), Article ISSTA035.
https://doi.org/10.1145/3728910

van Wyk, F., Wang, Y., Khojandi, A., & Masoud, N. (2020). Real-time sensor
anomaly detection and identification in automated vehicles. *IEEE
Transactions on Intelligent Transportation Systems*, *21*(3), 1264-1276.
https://doi.org/10.1109/TITS.2019.2906038

Werling, M., Faller, R., Betz, W., & Straub, D. (2025). Safety integrity
framework for automated driving. *arXiv preprint arXiv:2503.20544*.
https://doi.org/10.48550/arXiv.2503.20544

Levin, M. Sh. (2012). Morphological methods for design of modular systems:
A survey. *arXiv preprint arXiv:1201.1712*.

Levin, M. Sh. (2015). *Modular System Design and Evaluation*. Springer.

Ritchey, T. (2015). Principles of cross-consistency assessment in general
morphological modelling. *Acta Morphologica Generalis*, *4*(2).

Roy, B. (1996). *Multicriteria Methodology for Decision Aiding*.
Kluwer Academic Publishers.


### Processo de avaliacao -- LLMs

#### Debate estruturado -- CCM E5 (escala {0, 0.5, 1.0})

| LLM | Link |
|---|---|
| Claude | https://claude.ai/share/5f265562-909f-4b60-af15-a6c738b30a71 |
| Gemini | https://gemini.google.com/share/ae86b4a76e6e |
| ChatGPT | https://chatgpt.com/share/69df8e68-2028-83e9-ba2b-c31e4ec4ac62 |
| DeepSeek | https://chat.deepseek.com/share/tltr4h05pjoxipg0xn |

